In [1]:
from sympy import *
%run Geom_Prolongation.ipynb
%run Particular_Distributions.ipynb
%run CartanGeometry.ipynb

In [2]:
g=Symp_symb(7)
C=g.cochain_complex
K=IndexedBase('K')
Y,H,E,X,e1,e2,e3,e4,e5,e6,N=g.basis
P=RegularCartanGeometry(g,'eta')
D=Distr_of_constant_symbol(g,-P.curvature)
eta=IndexedBase('eta')
P.fund_invars=[eta[3,9,6],eta[6,9,9],eta[3,9,4]]

In [6]:
tuples_by_wght={}
for i in range(3,len(g.basis)):
    for j in range(i+1,len(g.basis)):
        for k in range(j+1,len(g.basis)):
            for m in range(len(g.basis)):
                w=-g.basis[i].wght-g.basis[j].wght-g.basis[k].wght+g.basis[m].wght
                if w not in tuples_by_wght: tuples_by_wght[w]=[]
                tuples_by_wght[w].append((i,j,k,m))

### Computations

In [7]:
Bianchi_dict={}
not_added=[]

def compute_Bianchi(w):
    for t in tuples_by_wght[w]:
        i,j,k,m=t
        time0=time.time()
        print('Computing', t)
        if (i,j,k) in P.Bianchi_cache:
            zero_elt=P.Bianchi_cache[(i,j,k)]
        else: 
            time1=time.time()
            zero_elt=Bianchi(P,i,j,k)
            P.Bianchi_cache[(i,j,k)]=zero_elt
            print('    Bianchi computed in',hrs_min_sec(time.time()-time1))
        time2=time.time()
        to_solve=ds_subs(zero_elt.vec[m],Bianchi_dict,D)
        print('    to_solve computed in',hrs_min_sec(time.time()-time2))
        if simplify(to_solve)!=0:
            time3=time.time()
            s=find_a_linear_term(to_solve,P.fund_invars)
            if s==None: s=find_a_linear_term(to_solve)
            if s==None:
                print('no linear term in',t)
                not_added.append(t)
            else:
                sol=solve(to_solve,s,dict=True)[0]
                print('    Solving complete in',hrs_min_sec(time.time()-time3))
                time4=time.time()
                for a in sol: 
                    ds_add_key(a,sol[a],Bianchi_dict,D)
                print('    Substitution complete in',hrs_min_sec(time.time()-time4))
        print('   ',t,'computed in',hrs_min_sec(time.time()-time0))
        # Notice that D.curv = -P.curvature, since I switched sign conventions

def check_Bianchi(w):
    for t in tuples_by_wght[w]:
        i,j,k,m=t
        if (i,j,k) in P.Bianchi_cache:
            zero_elt=P.Bianchi_cache[(i,j,k)]
        else:
            zero_elt=Bianchi(P,i,j,k)
            P.Bianchi_cache[(i,j,k)]=zero_elt
        r=simplify(ds_subs(zero_elt.vec[m],Bianchi_dict,D))
        if r!=0: print(r)

In [8]:
for w in range(1,10):
    time0=time.time()
    print('--------------------- Weight',w,'---------------------')
    compute_Bianchi(w)
    print('Bianchi wght',w,'computed in', hrs_min_sec(time.time()-time0))
    # In order to track what identities are being used, I won't want to substitute here. It takes longer though :/
    # P.update_fund_ders(Bianchi_dict,D)
    # P.curvature=ds_subs(P.curvature,Bianchi_dict,D)
    # D.curv=-P.curvature
    print('Weight',w,'complete in',hrs_min_sec(time.time()-time0))

--------------------- Weight 1 ---------------------
Computing (3, 4, 5, 6)
    Bianchi computed in 3.0 sec
    to_solve computed in 0.0 sec
    Solving complete in 0.0 sec
    Substitution complete in 0.0 sec
    (3, 4, 5, 6) computed in 3.0 sec
Computing (3, 4, 6, 7)
    Bianchi computed in 2.0 sec
    to_solve computed in 0.0 sec
    Solving complete in 0.0 sec
    Substitution complete in 0.0 sec
    (3, 4, 6, 7) computed in 2.0 sec
Computing (3, 4, 7, 8)
    Bianchi computed in 4.0 sec
    to_solve computed in 0.0 sec
    Solving complete in 0.0 sec
    Substitution complete in 0.0 sec
    (3, 4, 7, 8) computed in 4.0 sec
Computing (3, 4, 8, 9)
    Bianchi computed in 7.0 sec
    to_solve computed in 0.0 sec
    Solving complete in 0.0 sec
    Substitution complete in 0.0 sec
    (3, 4, 8, 9) computed in 7.0 sec
Computing (3, 4, 9, 10)
    Bianchi computed in 22.0 sec
    to_solve computed in 0.0 sec
    Solving complete in 0.0 sec
    Substitution complete in 0.0 sec
    (3, 4, 9

In [9]:
print(ds_subs_needed(Bianchi_dict))
print(not_added)

False
[]


### Plugging in Wilc=0

In [10]:
zero_wilc_dict=copy.deepcopy(Bianchi_dict)

In [11]:
zero_wilc_dict[eta[6,9,9]]

{(4, 4): 0,
 (3, 4, 4): 0,
 (4, 3, 4): 0,
 (4,
  3,
  3,
  4): 3*eta[6, 9, 9, 3, 4, 3, 4]/11 - 84*eta[6, 9, 9, 4]*eta[6, 9, 9]/55,
 (3, 3, 4, 3): eta[3, 9, 6, 3, 4, 4]/7 + 2*eta[6, 9, 9, 3, 3, 3, 4],
 (3, 3, 4, 4): 2*eta[6, 9, 9, 3, 4, 3, 4] - 4*eta[6, 9, 9, 4]*eta[6, 9, 9]/15,
 (3,
  4,
  3,
  4,
  3): eta[3, 9, 6, 3, 4, 4, 4]/7 + 3*eta[6, 9, 9, 3, 3, 3, 4, 4]/2 + 2*eta[6, 9, 9, 3]*eta[6, 9, 9, 4]/15 + 2*eta[6, 9, 9, 4, 3]*eta[6, 9, 9]/15,
 (3,
  4,
  3,
  3,
  4): 93*eta[3, 9, 6, 3, 4, 4, 4]/616 + 39*eta[6, 9, 9, 3, 3, 3, 4, 4]/22 - 6*eta[6, 9, 9, 3, 4]*eta[6, 9, 9]/5 - 194*eta[6, 9, 9, 3]*eta[6, 9, 9, 4]/55 + 37*eta[6, 9, 9, 4, 3]*eta[6, 9, 9]/55,
 (4,
  3,
  3,
  3,
  4): 87*eta[3, 9, 6, 3, 4, 4, 4]/1232 + 17*eta[6, 9, 9, 3, 3, 3, 4, 4]/22 - 4*eta[6, 9, 9, 3, 4]*eta[6, 9, 9]/5 - 282*eta[6, 9, 9, 3]*eta[6, 9, 9, 4]/55 - 329*eta[6, 9, 9, 4, 3]*eta[6, 9, 9]/165,
 (3,
  3,
  3,
  3,
  3,
  4): eta[3, 9, 4, 3, 4, 4]/264 - eta[3, 9, 4, 4, 4, 3]/44 + eta[3, 9, 6, 3, 3, 3, 4, 4]/21 + 5*eta

In [12]:
ds_add_key(eta[3,9,6],0,zero_wilc_dict,D)
ds_add_key(eta[3,9,4],0,zero_wilc_dict,D)

In [13]:
print(ds_subs_needed(zero_wilc_dict))

False


In [15]:
factor(not_added[0])

-624834*(6*eta[6, 9, 9, 3, 4] - eta[6, 9, 9, 4, 3])*eta[6, 9, 9, 4]/4300015

In [16]:
not_added_bookmark=copy.copy(not_added)

#### First Branch

In [17]:
b1_Bianchi_dict=copy.deepcopy(zero_wilc_dict)
not_added=copy.copy(not_added_bookmark)
ds_add_key(eta[6,9,9,4],0,b1_Bianchi_dict,D)

In [18]:
for a in not_added:
    display(ds_subs(a,b1_Bianchi_dict,D))

0

4*eta[6, 9, 9, 3, 4]*eta[6, 9, 9]/5

36*eta[6, 9, 9, 3, 4, 3]*eta[6, 9, 9]/5 - 96*eta[6, 9, 9, 3, 4]*eta[6, 9, 9, 3]/5 + 36*eta[6, 9, 9]**3/25

In [19]:
ds_add_key(eta[6,9,9,3,4],0,b1_Bianchi_dict,D)

In [20]:
for a in not_added:
    display(ds_subs(a,b1_Bianchi_dict,D))

0

0

36*eta[6, 9, 9]**3/25

6*eta[6, 9, 9]**3/25

#### Second Branch

In [21]:
not_added=copy.copy(not_added_bookmark)
factor(not_added[0])

-624834*(6*eta[6, 9, 9, 3, 4] - eta[6, 9, 9, 4, 3])*eta[6, 9, 9, 4]/4300015

In [22]:
b2_Bianchi_dict=copy.deepcopy(zero_wilc_dict)
ds_add_key(eta[6,9,9,4,3],6*eta[6,9,9,3,4],b2_Bianchi_dict,D)

In [23]:
not_added[1]

126*eta[6, 9, 9, 4]*eta[6, 9, 9]/55

In [24]:
ds_add_key(eta[6,9,9,4],0,b2_Bianchi_dict,D)

In [25]:
not_added[-1]

6*eta[6, 9, 9]**3/25

### Continuing

In [53]:
# with shelve.open('Abstract_Syzygies') as shelf:
#     shelf['Bianchi_dict'+'{j}'.format(j=7)]=Bianchi_dict

In [54]:
saved_Bianchi_dict=copy.deepcopy(Bianchi_dict) # Through wght 14 at the moment
saved_curv=copy.deepcopy(P.curvature)

In [55]:
# with shelve.open('Abstract_Syzygies') as shelf:
#     temp=shelf['Bianchi_dict'+'{j}'.format(j=7)]

In [ ]:
for a in P.fund_invars:
    print(a, a in Bianchi_dict)

In [57]:
# Bianchi_dict=copy.deepcopy(saved_Bianchi_dict) # Through wght 13 at the moment
# P.curvature=copy.deepcopy(saved_curv)

In [58]:
P.curvature=ds_subs(P.curvature,Bianchi_dict,D)
D.curv=-P.curvature

In [ ]:
for w in range(1,10):
    print(w)
    check_Bianchi(w)

In [ ]:
# These should only be derivatives of the fundamental invariants
temp=set()
for k1 in Bianchi_dict:
    for k2 in Bianchi_dict[k1]:
        temp=temp.union(Indexed_obj_in_expr(Bianchi_dict[k1][k2]))
temp

In [61]:
syzygies_by_wght={}
for k in P.fund_invars:
    for j in Bianchi_dict[k]:
        w=-wght_of_ind(k.base[k.indices+j],g)
        if not w in syzygies_by_wght: syzygies_by_wght[w]=[]
        new_syzygy=simplify(k.base[k.indices+j]-Bianchi_dict[k][j])
        if new_syzygy!=0:
            new_syzygy=new_syzygy*new_syzygy.as_numer_denom()[1]
            syzygies_by_wght[w].append(new_syzygy)

In [63]:
zero_Wilc_dict={eta[3,9,6]:{tuple():0},eta[3,9,4]:{tuple():0}}

In [ ]:
for a in Bianchi_dict[eta[6,9,9]]:
    print(a,'-->',Bianchi_dict[eta[6,9,9]][a])
    print()

In [ ]:
for a in syzygies_by_wght[9]:
    display('-------------------------')
    display(ds_subs(a,zero_Wilc_dict,D))
    display(ds_subs(ds_subs(ds_subs(a,zero_Wilc_dict,D),Bianchi_dict,D),zero_Wilc_dict,D))

In [12]:
I=IndexedBase('I')
W=IndexedBase('W')

fund_invar_dict={I[3]:eta[6,9,9],W[4]:eta[3,9,6],W[6]:eta[3,9,4]}

def convert_syzygy(syz):
    T=Indexed_obj_in_expr(syz)
    s={}
    for t in T:
        temp=fund_invar_dict[t.base[t.indices[0]]]
        s[t]=temp.base[temp.indices+t.indices[1:len(t.indices)]]
    return syz.xreplace(s)

In [ ]:
old_syzygies=[
    7*I[3, 3, 3] + W[4, 4],
    14*I[3, 3, 3, 3, 4] - 7*I[3, 3, 3, 4, 3] + W[4, 3, 4, 4],
    -7*I[3, 3, 3, 3, 4, 4] + 14*I[3, 3, 3, 4, 3, 4] - 7*I[3, 3, 3, 4, 4, 3],
    -7*I[3, 3, 3, 3, 4, 4] + 14*I[3, 3, 3, 4, 3, 4] - 7*I[3, 3, 3, 4, 4, 3],
    -28*I[3, 3, 3, 3, 3, 3]/5 - 96*I[3, 3]*W[4]/5 - 12*I[3]*W[4, 3]/5 - 7*W[4, 3, 3, 3, 4]/30 + W[4, 3, 3, 4, 3] - 3*W[4, 3, 4, 3, 3]/2 + 49*W[6, 3, 4]/1650 - 14*W[6, 4, 3]/275
]

for old_syz in old_syzygies:
    print(simplify(ds_subs(convert_syzygy(old_syz),Bianchi_dict,D)))

In [ ]:
C.subspace_proj(ds_subs(D.curv.wght_proj(4),Bianchi_dict,D),'harmonic')
C.subspace_proj(ds_subs(D.curv.wght_proj(4),Bianchi_dict,D),'coexact')

In [ ]:
ds_subs(P.fund_der(eta[6,9,9],3),Bianchi_dict,D)

In [29]:
partial_Bianchi_dicts={20:copy.deepcopy(Bianchi_dict)}
for w in reversed(range(20)):
    r=copy.deepcopy(partial_Bianchi_dicts[w+1])
    for k in r:
        for j in list(r[k].keys()):
            if -wght_of_ind(k.base[k.indices+j],g)>w: r[k].pop(j)
    for k in list(r.keys()):
        if r[k]==dict(): r.pop(k)
    partial_Bianchi_dicts[w]=r

In [ ]:
harm_basis={}
for i in [3,4,6]:
    harm_basis[i]=C.elt({})
    for j in range(len(C.basis(2,i))):
        harm_basis[i]=harm_basis[i]+C.subspace_basis('harmonic',2,i)[j]*C.basis(2,i)[j]
